# RaceShift FFR Colab Training

Train and evaluate the RaceShift **Forward-Forward regressor** (no global backpropagation) on real Formula 1 laps.

How this notebook is organised:

1. Environment check and Google Drive mount. Code lives in `/content/RaceShift`; data, caches and artifacts persist in `Drive/RaceShiftData`.
2. Clone (or update) the project from GitHub and install it.
3. **Synthetic smoke test first.** It proves the pipeline before any race data is downloaded. Its numbers are never Formula 1 claims.
4. Collect a small real subset: 2022-2025 races at Bahrain, Silverstone and Monza.
5. **Baselines before the neural model.** Previous lap, rolling-five median, ridge and gradient boosting on the same leakage-safe table.
6. Train the production FFR ladder (512 → 384 → 256 → 192, 8/16/32/64 ordinal groups) with a chronological split: train ≤ 2023, validate 2024, test 2025.
7. Optional wider ablation and Hugging Face source inspection.
8. Copy the artifact back into the local app.

RaceShift FFR is NumPy-based. A GPU runtime is **optional**; a CPU or high-RAM runtime is fine. Colab does not guarantee runtime duration, so everything important is written to Drive as soon as it is produced.

In [ ]:
import sys, platform
print('Python:', sys.version.split()[0], '|', platform.platform())
try:
    import torch  # only informational: RaceShift FFR does not use torch for training
    print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except Exception as exc:  # noqa: BLE001
    print('torch not available (fine for RaceShift FFR):', exc)

## 1. Mount Google Drive

Everything under `RaceShiftData` survives runtime resets: the FastF1 cache, raw parquet files, processed tables and trained artifacts.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RaceShiftData')
RAW = DRIVE_ROOT / 'raw' / 'fastf1'
CACHE = DRIVE_ROOT / 'cache' / 'fastf1'
PROCESSED = DRIVE_ROOT / 'processed'
ARTIFACTS = DRIVE_ROOT / 'artifacts'
for p in (RAW, CACHE, PROCESSED, ARTIFACTS):
    p.mkdir(parents=True, exist_ok=True)
print('Persistent data root:', DRIVE_ROOT)

## 2. Get the RaceShift code

The project lives in GitHub. Change `REPO` / `BRANCH` if you are working from a fork or a feature branch. Re-running this cell pulls the latest commit.

In [ ]:
import os, subprocess
REPO = 'https://github.com/parthd25/raceshift'
BRANCH = 'main'
PROJECT = Path('/content/RaceShift')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, str(PROJECT)], check=True)
os.chdir(PROJECT)
print(subprocess.run(['git', 'log', '--oneline', '-n', '1'], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q -e '.[research]'
import raceshift
print('raceshift', raceshift.__version__)

## 3. Synthetic smoke test (no race data yet)

This verifies `import → features → chronological split → local Forward-Forward training → artifact → evaluation` in under a minute. The metrics describe a synthetic fixture and must never be quoted as Formula 1 performance.

In [ ]:
SMOKE = Path('/content/smoke')
SMOKE.mkdir(exist_ok=True)
!python scripts/make_synthetic_fixture.py --output {SMOKE}/synthetic_fixture.csv
!python scripts/run_experiments.py --input {SMOKE}/synthetic_fixture.csv --name synthetic_smoke \
  --train-end 2023 --val-year 2024 --test-year 2025 --ffr configs/ffr_demo.json \
  --artifacts-dir {SMOKE}/artifacts --reports-dir {SMOKE}/reports

## 4. Collect real Formula 1 races

Twelve circuits across 2022-2026 give a wide mix of layouts, weather and altitude. `--events`
matches circuit locations as well as Grand Prix names. FastF1 caches every session in Drive,
so re-runs are fast, and the public F1 API allows about 500 calls per hour: if you see
`RateLimitExceededError`, wait for the window to reset and run the cell again (cached sessions
are skipped instantly).

In [ ]:
!python scripts/fetch_fastf1_seasons.py \
  --years 2022-2026 \
  --session R \
  --events Bahrain Jeddah Suzuka Monaco Silverstone Spa Monza "Marina Bay" Austin "Mexico City" "São Paulo" "Yas Island" \
  --output "{RAW}" \
  --cache "{CACHE}"

In [ ]:
import glob
import pandas as pd
files = sorted(glob.glob(str(RAW / '*.parquet')))
laps = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
LAPS = PROCESSED / 'f1_laps.parquet'
laps.to_parquet(LAPS, index=False)
print(len(files), 'sessions,', len(laps), 'raw laps ->', LAPS)
laps.groupby('season')['event'].nunique()

## 5. Run the experiment matrix

`scripts/run_experiments.py` runs the required baselines (previous lap, rolling-five median,
ridge, gradient-boosted trees) and each Forward-Forward ladder **on the same leakage-safe
table and split**, as separate processes so peak memory is measured per run, then writes
`reports/<name>/summary.md`.

Split used here: train 2022-2024, validation = 2025 rounds 1-12, test = 2025 rounds 13+.
Baselines run first; the FFR ladders follow.

In [ ]:
!python scripts/run_experiments.py \
  --input "{LAPS}" --name f1_2025h2 \
  --train-end 2024 --val-year 2025 --test-year 2025 --split-round 12 \
  --ffr configs/ffr_small.json configs/ffr_production.json configs/ffr_colab_large.json \
        configs/ffr_m_groups_coarse.json configs/ffr_m_groups_fine.json \
  --ablate historical_numeric temporal_numeric static_categorical \
  --artifacts-dir "{ARTIFACTS}" --reports-dir "{DRIVE_ROOT}/reports"

In [ ]:
from IPython.display import Markdown, display
display(Markdown((DRIVE_ROOT / 'reports' / 'f1_2025h2' / 'summary.md').read_text()))

## 6. Circuit holdout and the 2026 domain-shift test

Circuit holdout: every season of one circuit is held out. Domain shift: train through 2025 and
test on the 2026 rounds collected above (skip if no 2026 sessions were downloaded yet).

In [ ]:
!python scripts/run_experiments.py --input "{LAPS}" --name holdout_monza \
  --train-end 2024 --val-year 2025 --test-year 2025 --holdout-event "Italian Grand Prix" \
  --ffr configs/ffr_production.json --artifacts-dir "{ARTIFACTS}" --reports-dir "{DRIVE_ROOT}/reports"

In [ ]:
if (laps['season'] == 2026).any():
    !python scripts/run_experiments.py --input "{LAPS}" --name domain_shift_2026 \
      --train-end 2024 --val-year 2025 --test-year 2026 \
      --ffr configs/ffr_small.json configs/ffr_production.json --artifacts-dir "{ARTIFACTS}" --reports-dir "{DRIVE_ROOT}/reports"
else:
    print('No 2026 sessions collected yet; re-run the collection cell later in the season.')

## 7. Optional: feature ablations already ran above

The `--ablate` flags drop one feature group at a time (historical priors, temporal context,
static identity) from the first FFR config so the report answers "what information helped?",
not only "what scored best?".

## 8. Optional: inspect large Hugging Face sources before downloading

These datasets are multi-gigabyte. Listing is free; downloading is opt-in and stays outside git. Check each source's licence in `docs/DATA_AND_MODEL_FINDINGS.md` first.

In [ ]:
!python scripts/fetch_hf.py FlorindoDev/f1_corner_telemetry_2024_2025 --list
!python scripts/fetch_hf.py VforVitorio/f1-strategy-dataset --list
!python scripts/fetch_hf.py tobil/imsa --list

## 9. Use the trained artifact locally

Download the folder `RaceShiftData/artifacts/f1_2025h2_ffr-m` (six files plus `test_predictions.csv`) and copy it into the project's `artifacts/` directory on your machine, for example `artifacts/f1_2025h2_ffr-m/`. The local API lists every complete artifact under `GET /api/models`, and the Forecast page lets you pick it. Real artifacts are labelled by their `data_source`; the packaged `raceshift_ffr_demo` stays labelled synthetic.

Reporting rules: quote future-season test metrics with the holdout year named, always next to the baselines from step 5, and never quote step-3 synthetic numbers as Formula 1 results.

## Full unattended pipeline (every team, every round, both tiers)

`scripts/full_pipeline.sh` collects the FastF1 tier (2018 → today) and the legacy Jolpica/Ergast tier (2000-2017), builds the tables, and runs the main matrix, the Monza circuit holdout, the 2026 domain-shift run and the legacy training-set extension. Every stage is resumable, and both collectors wait out the public API budgets (about 500 requests per hour each), so a complete run takes several hours. Keep the Colab tab open or run stages one at a time.

In [ ]:
%%bash
cd /content/RaceShift
export RAW_FF=/content/drive/MyDrive/RaceShiftData/raw/fastf1 \
       RAW_LEGACY=/content/drive/MyDrive/RaceShiftData/raw/jolpica \
       CACHE_FF=/content/drive/MyDrive/RaceShiftData/cache \
       PROC=/content/drive/MyDrive/RaceShiftData/processed \
       ARTIFACTS=/content/drive/MyDrive/RaceShiftData/artifacts \
       REPORTS=/content/drive/MyDrive/RaceShiftData/reports
bash scripts/full_pipeline.sh fastf1 build-fastf1 experiments
# bash scripts/full_pipeline.sh legacy build-all legacy-experiments   # optional: 2000-2017 extension
